In [ ]:
import os,glob,shutil,random,json,time,warnings
from concurrent.futures import ThreadPoolExecutor,as_completed

import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt
import seaborn as sns

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report,confusion_matrix

warnings.filterwarnings("ignore",category=UserWarning)
cv2.setNumThreads(1)

print("TensorFlow version:",tf.__version__)
gpus=tf.config.list_physical_devices('GPU')
print("GPU available:",gpus)

for g in gpus:
    try:
        tf.config.experimental.set_memory_growth(g,True)
    except Exception:
        pass

try:
    keras.mixed_precision.set_global_policy("mixed_float16")
    print("Mixed precision enabled (mixed_float16).")
except Exception as e:
    print("Mixed precision not enabled:",e)

In [ ]:
DATA_DIR="/kaggle/input/datasets/mdfaysalhossan/agri-27-dataset/Agri 27 Dataset"
PRE_SPLIT=True

MODEL_NAME="AgriNet-27"

OUTPUT_DIR="/kaggle/working/split_data"
CKPT_DIR="/kaggle/working/_checkpoints"
RUN_DIR=os.path.join("/kaggle/working/runs",MODEL_NAME)
for _d in (OUTPUT_DIR,CKPT_DIR,RUN_DIR):
    os.makedirs(_d,exist_ok=True)

SELECTED_CLASSES=None
MAX_IMAGES_PER_CLASS=None

DO_HAIR_REMOVAL=False
DO_DENOISE=False
DO_CLAHE=True

IMG_SIZE=224
DROPOUT_RATE=0.2

EPOCHS=1
LR=1e-4
LABEL_SMOOTHING=0.05
CLIPNORM=1.0

BATCH_SIZE=64
SEED=42
CACHE_TRAIN_IN_RAM=False
IMAGE_EXTS=(".jpg",".jpeg",".png",".bmp",".tif",".tiff",".webp")

EFFICIENTNET_WEIGHTS_PATH=None

tf.random.set_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

print("MODEL_NAME:",MODEL_NAME)
print("DATA_DIR  :",DATA_DIR)
print("RUN_DIR   :",RUN_DIR)

In [ ]:
def _ckpt_dir():
    d = globals().get("CKPT_DIR", "/kaggle/working/_checkpoints")
    os.makedirs(d, exist_ok=True)
    return d

def stage_done(name):
    return os.path.exists(os.path.join(_ckpt_dir(), name + ".done"))

def mark_done(name, info=None):
    with open(os.path.join(_ckpt_dir(), name + ".done"), "w") as f:
        json.dump(info or {}, f)
    print(f"[checkpoint] stage '{name}' marked complete.")

def load_done(name):
    p = os.path.join(_ckpt_dir(), name + ".done")
    return json.load(open(p)) if os.path.exists(p) else None

def save_json(obj, path):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, "w") as f:
        json.dump(obj, f)

def load_json(path):
    return json.load(open(path)) if os.path.exists(path) else None

def with_retries(fn, retries=10, base_delay=5, max_delay=120, what="operation"):
    """Run fn() with exponential-backoff retries -- survives brief internet drops."""
    for attempt in range(1, retries + 1):
        try:
            return fn()
        except Exception as e:
            if attempt == retries:
                print(f"[retry] {what}: giving up after {retries} attempts.")
                raise
            delay = min(max_delay, base_delay * (2 ** (attempt - 1)))
            print(f"[retry] {what} failed (attempt {attempt}/{retries}): {e}")
            print(f"        reconnect the internet if it's off -- retrying in {delay}s ...")
            time.sleep(delay)

print("Resume/checkpoint utilities ready.")


In [ ]:
CLASS_NAMES_PATH=os.path.join(CKPT_DIR,"class_names.json")

def discover_classes(data_dir):
    leaves={}
    for root,dirs,files in os.walk(data_dir):
        imgs=[os.path.join(root,f) for f in files if f.lower().endswith(IMAGE_EXTS)]
        if not imgs:
            continue
        child_has_imgs=False
        for d in dirs:
            dp=os.path.join(root,d)
            try:
                if any(x.lower().endswith(IMAGE_EXTS) for x in os.listdir(dp)):
                    child_has_imgs=True
                    break
            except Exception:
                pass
        if not child_has_imgs:
            leaves[root]=imgs

    name_counts={}
    for p in leaves:
        b=os.path.basename(p)
        name_counts[b]=name_counts.get(b,0)+1

    classes={}
    for p,imgs in leaves.items():
        base=os.path.basename(p)
        if name_counts[base]>1:
            base=f"{os.path.basename(os.path.dirname(p))}_{base}"
        classes[base]=imgs
    return classes


def build_split(data_dir,output_dir,selected=None,max_per_class=None,
                train_ratio=0.8,val_ratio=0.1,test_ratio=0.1,seed=42):
    assert os.path.isdir(data_dir),(
        f"DATA_DIR not found: {data_dir}\n-> Attach the dataset and fix DATA_DIR in Config."
    )
    classes=discover_classes(data_dir)
    assert classes,f"No image leaf-folders found under {data_dir}"

    if selected:
        missing=[c for c in selected if c not in classes]
        if missing:
            print("WARNING: these SELECTED_CLASSES were not found:",missing)
        classes={c:classes[c] for c in selected if c in classes}
        assert classes,"None of the SELECTED_CLASSES matched discovered folders."

    class_names=sorted(classes.keys())
    print(f"Discovered {len(class_names)} classes.\n")

    rng=random.Random(seed)
    for split in ["train","val","test"]:
        for c in class_names:
            os.makedirs(os.path.join(output_dir,split,c),exist_ok=True)

    for c in class_names:
        files=sorted(classes[c])
        if max_per_class and len(files)>max_per_class:
            files=rng.sample(files,max_per_class)
        train_files,temp=train_test_split(files,train_size=train_ratio,random_state=seed)
        rel_val=val_ratio/(val_ratio+test_ratio)
        val_files,test_files=train_test_split(temp,train_size=rel_val,random_state=seed)

        for f in train_files:
            shutil.copy(f,os.path.join(output_dir,"train",c,os.path.basename(f)))
        for f in val_files:
            shutil.copy(f,os.path.join(output_dir,"val",c,os.path.basename(f)))
        for f in test_files:
            shutil.copy(f,os.path.join(output_dir,"test",c,os.path.basename(f)))

        print(f"{c:28s} -> train:{len(train_files):5d}  val:{len(val_files):5d}  test:{len(test_files):5d}")
    return class_names


if PRE_SPLIT:
    SPLIT_DIR=DATA_DIR

    def _resolve_split(root,aliases):
        for a in aliases:
            p=os.path.join(root,a)
            if os.path.isdir(p):
                return p,a
        raise AssertionError(
            f"PRE_SPLIT=True but none of {aliases} found under DATA_DIR:\n  {root}\n"
            f"Folders present: {os.listdir(root) if os.path.isdir(root) else 'NOT A DIRECTORY'}\n"
            f"-> point DATA_DIR to the folder that DIRECTLY contains the train/val/test folders."
        )

    TRAIN_DIR,_tn=_resolve_split(DATA_DIR,["train","Train","TRAIN","training","Training"])
    VAL_DIR,_vn=_resolve_split(DATA_DIR,["val","Val","valid","Valid","validation","Validation","VALIDATION"])
    TEST_DIR,_en=_resolve_split(DATA_DIR,["test","Test","TEST","testing","Testing"])

    CLASS_NAMES=sorted(
        d for d in os.listdir(TRAIN_DIR)
        if os.path.isdir(os.path.join(TRAIN_DIR,d))
    )

    save_json(CLASS_NAMES,CLASS_NAMES_PATH)
    print(f"[pre-split] Using existing splits -> {_tn}/ {_vn}/ {_en}/ under DATA_DIR (no re-split).")

else:
    SPLIT_DIR=OUTPUT_DIR
    TRAIN_DIR=os.path.join(SPLIT_DIR,"train")
    VAL_DIR=os.path.join(SPLIT_DIR,"val")
    TEST_DIR=os.path.join(SPLIT_DIR,"test")

    if stage_done("split"):
        CLASS_NAMES=load_json(CLASS_NAMES_PATH)
        print("[resume] split already done -- skipping copy. Classes:",len(CLASS_NAMES))
    else:
        CLASS_NAMES=build_split(
            DATA_DIR,OUTPUT_DIR,SELECTED_CLASSES,MAX_IMAGES_PER_CLASS,
            0.8,0.1,0.1,SEED
        )
        save_json(CLASS_NAMES,CLASS_NAMES_PATH)
        mark_done("split",{"classes":CLASS_NAMES})

NUM_CLASSES=len(CLASS_NAMES)
print("\nClasses:",CLASS_NAMES)
print("Number of classes:",NUM_CLASSES)

In [ ]:
DO_DENOISE=True
DO_CLAHE=True
DO_GAMMA=True
DO_NORMALIZE=True
GAMMA=1.5

def denoise_image(bgr):
    return cv2.fastNlMeansDenoisingColored(bgr,None,10,10,7,21)

def apply_clahe(bgr):
    lab=cv2.cvtColor(bgr,cv2.COLOR_BGR2LAB)
    l,a,b=cv2.split(lab)
    l=cv2.createCLAHE(clipLimit=2.0,tileGridSize=(8,8)).apply(l)
    return cv2.cvtColor(cv2.merge((l,a,b)),cv2.COLOR_LAB2BGR)

def gamma_correction(bgr,gamma=1.5):
    inv=1.0/max(gamma,1e-6)
    table=(((np.arange(256)/255.0)**inv)*255).astype("uint8")
    return cv2.LUT(bgr,table)

def normalize_image(bgr):
    f=bgr.astype(np.float32)
    mn,mx=float(f.min()),float(f.max())
    if mx-mn<1e-6:
        return bgr
    return np.clip((f-mn)/(mx-mn)*255.0,0,255).astype(np.uint8)

def apply_preprocessing_bgr(bgr):
    if DO_DENOISE:
        bgr=denoise_image(bgr)
    if DO_CLAHE:
        bgr=apply_clahe(bgr)
    if DO_GAMMA:
        bgr=gamma_correction(bgr,GAMMA)
    if DO_NORMALIZE:
        bgr=normalize_image(bgr)
    return bgr

_active=" -> ".join(
    (["Denoise(h=10)"] if DO_DENOISE else [])+
    (["CLAHE(2.0,8x8)"] if DO_CLAHE else [])+
    ([f"Gamma({GAMMA})"] if DO_GAMMA else [])+
    (["MinMaxNorm[0,255]"] if DO_NORMALIZE else [])
)
print("Preprocessing (on-the-fly in loader):",_active if _active else "(none)")

In [ ]:
import matplotlib.pyplot as plt

def _find_raw_sample(root):
    for r,_,files in os.walk(root):
        for f in sorted(files):
            if f.lower().endswith(IMAGE_EXTS):
                return os.path.join(r,f)
    return None

_sp=_find_raw_sample(DATA_DIR)
assert _sp is not None,f"No image found under DATA_DIR: {DATA_DIR}"

_bgr=cv2.resize(cv2.imread(_sp),(IMG_SIZE,IMG_SIZE),interpolation=cv2.INTER_AREA)

def _rgb(b):
    return cv2.cvtColor(b,cv2.COLOR_BGR2RGB)

den=_rgb(denoise_image(_bgr))
cla=_rgb(apply_clahe(_bgr))
gam=_rgb(gamma_correction(_bgr,GAMMA))
nrm=_rgb(normalize_image(_bgr))
final=_rgb(apply_preprocessing_bgr(_bgr.copy()))

def _tag(name,on):
    return f"{name}\n(used: {'yes' if on else 'no'})"

panels=[
    ("Original (resized)",_rgb(_bgr)),
    (_tag("1. Denoise (NLM)",DO_DENOISE),den),
    (_tag("2. CLAHE",DO_CLAHE),cla),
    (_tag(f"3. Gamma ({GAMMA})",DO_GAMMA),gam),
    (_tag("4. Normalization",DO_NORMALIZE),nrm),
    ("Final (pipeline output)",final)
]

n=len(panels)
fig,axes=plt.subplots(1,n,figsize=(3.3*n,4.2))

for ax,(title,im) in zip(axes,panels):
    ax.imshow(im)
    ax.set_title(title,fontsize=15)
    ax.axis("off")

plt.tight_layout()
_out=os.path.join(RUN_DIR,"preprocessing_steps.png")
plt.savefig(_out,dpi=300,bbox_inches="tight")
plt.show()

print("sample image :",_sp)
print("active pipeline:"," -> ".join(
    (["Denoise"] if DO_DENOISE else [])+
    (["CLAHE"] if DO_CLAHE else [])+
    ([f"Gamma({GAMMA})"] if DO_GAMMA else [])+
    (["Normalize"] if DO_NORMALIZE else [])
))
print("saved figure :",_out)

In [ ]:
def _resolve_split(root,aliases):
    for a in aliases:
        p=os.path.join(root,a)
        if os.path.isdir(p):
            return p
    raise FileNotFoundError(
        f"None of {aliases} found under:\n  {root}\n"
        f"Folders present: {os.listdir(root) if os.path.isdir(root) else 'NOT A DIRECTORY'}"
    )

_base=SPLIT_DIR if "SPLIT_DIR" in globals() else DATA_DIR
train_dir=TRAIN_DIR if "TRAIN_DIR" in globals() else _resolve_split(_base,["train","Train","TRAIN","training","Training"])
val_dir=VAL_DIR if "VAL_DIR" in globals() else _resolve_split(_base,["val","Val","valid","Valid","validation","Validation","VALIDATION"])
test_dir=TEST_DIR if "TEST_DIR" in globals() else _resolve_split(_base,["test","Test","TEST","testing","Testing"])

print("train_dir:",train_dir)
print("val_dir  :",val_dir)
print("test_dir :",test_dir)

train_ds=tf.keras.utils.image_dataset_from_directory(
    train_dir,
    image_size=(IMG_SIZE,IMG_SIZE),
    batch_size=BATCH_SIZE,
    label_mode="categorical",
    shuffle=True,
    seed=SEED
)

val_ds=tf.keras.utils.image_dataset_from_directory(
    val_dir,
    image_size=(IMG_SIZE,IMG_SIZE),
    batch_size=BATCH_SIZE,
    label_mode="categorical",
    shuffle=False
)

test_ds=tf.keras.utils.image_dataset_from_directory(
    test_dir,
    image_size=(IMG_SIZE,IMG_SIZE),
    batch_size=BATCH_SIZE,
    label_mode="categorical",
    shuffle=False
)

CLASS_NAMES=train_ds.class_names
NUM_CLASSES=len(CLASS_NAMES)
save_json(CLASS_NAMES,os.path.join(CKPT_DIR,"class_names.json"))

def _prep_batch_np(imgs):
    out=np.empty_like(imgs,dtype=np.float32)
    for i in range(imgs.shape[0]):
        bgr=cv2.cvtColor(imgs[i].astype(np.uint8),cv2.COLOR_RGB2BGR)
        bgr=apply_preprocessing_bgr(bgr)
        out[i]=cv2.cvtColor(bgr,cv2.COLOR_BGR2RGB).astype(np.float32)
    return out

def _prep_tf(x,y):
    x=tf.numpy_function(_prep_batch_np,[x],tf.float32)
    x.set_shape((None,IMG_SIZE,IMG_SIZE,3))
    return x,y

train_ds=train_ds.map(_prep_tf,num_parallel_calls=tf.data.AUTOTUNE)
val_ds=val_ds.map(_prep_tf,num_parallel_calls=tf.data.AUTOTUNE)
test_ds=test_ds.map(_prep_tf,num_parallel_calls=tf.data.AUTOTUNE)

AUTOTUNE=tf.data.AUTOTUNE
train_ds=(train_ds.cache().prefetch(AUTOTUNE)) if CACHE_TRAIN_IN_RAM else train_ds.prefetch(AUTOTUNE)
val_ds=val_ds.cache().prefetch(AUTOTUNE)
test_ds=test_ds.prefetch(AUTOTUNE)

print("Classes:",CLASS_NAMES)
print("Number of classes:",NUM_CLASSES)

In [ ]:
print("Per-split image counts:")
for _split,_d in [("train",train_dir),("val",val_dir),("test",test_dir)]:
    _total,_line=0,[]
    for _c in CLASS_NAMES:
        _cp=os.path.join(_d,_c)
        _n=len(os.listdir(_cp)) if os.path.isdir(_cp) else 0
        _total+=_n
        _line.append(f"{_c}:{_n}")
    print(f"  {_split:5s} total={_total:6d} | "+"  ".join(_line))

def _names(_d):
    _s=set()
    for _c in CLASS_NAMES:
        _cp=os.path.join(_d,_c)
        if os.path.isdir(_cp):
            _s|={f"{_c}/{_f}" for _f in os.listdir(_cp)}
    return _s

_tr=_names(train_dir)
_ov_te=_tr&_names(test_dir)
_ov_va=_tr&_names(val_dir)

print(f"\nLeakage check -> shared files  train/test: {len(_ov_te)}  |  train/val: {len(_ov_va)}")
print("  OK: no overlap." if not (_ov_te or _ov_va) else "  WARNING: overlap -- check the split!")
print("  Sanity: NUM_CLASSES =",NUM_CLASSES,"(should equal your real number of classes, e.g. 4)")

In [ ]:
def compute_class_weights(train_dir,class_names):
    counts={c:len(os.listdir(os.path.join(train_dir,c))) for c in class_names}
    total=sum(counts.values())
    n_classes=len(class_names)
    weights={i:total/(n_classes*counts[c]) for i,c in enumerate(class_names)}
    print("Train image counts per class:")
    for c in class_names:
        print(f"  {c:25s}: {counts[c]}")
    print("\nComputed class weights:")
    for i,c in enumerate(class_names):
        print(f"  {c:25s}: {weights[i]:.3f}")
    return weights

class_weight=compute_class_weights(train_dir,CLASS_NAMES)

In [ ]:
@tf.keras.utils.register_keras_serializable()
class RandomGaussianNoise(layers.Layer):
    def __init__(self,stddev=0.05,prob=0.5,**kw):
        super().__init__(**kw)
        self.stddev=stddev
        self.prob=prob

    def call(self,x,training=None):
        if not training:
            return x

        def _noisy():
            n=tf.random.normal(
                tf.shape(x),
                stddev=self.stddev*255.0,
                dtype=x.dtype
            )
            return tf.clip_by_value(x+n,0.0,255.0)

        return tf.cond(
            tf.random.uniform([])<self.prob,
            _noisy,
            lambda:x
        )

    def get_config(self):
        c=super().get_config()
        c.update(dict(stddev=self.stddev,prob=self.prob))
        return c

data_augmentation=keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomZoom(height_factor=(0.2,0.4),width_factor=(0.2,0.4)),
    layers.RandomRotation(30.0/360.0),
    layers.RandomTranslation(0.2,0.2),
    RandomGaussianNoise(stddev=0.05,prob=0.5),
    layers.RandomBrightness(0.3),
    layers.RandomContrast(0.3)
],name="train_augmentation")

preprocess_input=tf.keras.applications.efficientnet.preprocess_input

In [ ]:
import matplotlib.pyplot as plt

def _find_split_sample(root):
    for c in sorted(os.listdir(root)):
        cp=os.path.join(root,c)
        if os.path.isdir(cp):
            for f in sorted(os.listdir(cp)):
                if f.lower().endswith(IMAGE_EXTS):
                    return os.path.join(cp,f)
    return None

_ts=_find_split_sample(train_dir)
assert _ts is not None,f"No image found under train dir: {train_dir}"

_img=cv2.cvtColor(cv2.imread(_ts),cv2.COLOR_BGR2RGB)
_img=cv2.resize(_img,(IMG_SIZE,IMG_SIZE),interpolation=cv2.INTER_AREA)
_x=tf.convert_to_tensor(_img[None,...],dtype=tf.float32)

_techniques=[
    ("Horizontal Flip (p=0.5)",layers.RandomFlip("horizontal")),
    ("Zoom (scale 0.6-0.8)",layers.RandomZoom(height_factor=(0.2,0.4),width_factor=(0.2,0.4))),
    ("Rotation (+/-30 deg)",layers.RandomRotation(30.0/360.0)),
    ("Shift (+/-20%)",layers.RandomTranslation(0.2,0.2)),
    ("Gaussian Noise (p=0.5)",RandomGaussianNoise(stddev=0.05,prob=1.0)),
    ("Brightness (+/-30%)",layers.RandomBrightness(0.3)),
    ("Contrast (+/-30%)",layers.RandomContrast(0.3))
]

def _most_visible(layer,x,tries=8):
    base=x[0].numpy()
    best,best_d=None,-1.0
    for _ in range(tries):
        o=layer(x,training=True)[0].numpy()
        d=float(np.mean(np.abs(o-base)))
        if d>best_d:
            best_d,best=d,o
    return np.clip(best,0,255).astype("uint8")

panels=[("Original",_img)]
for _name,_layer in _techniques:
    panels.append((_name,_most_visible(_layer,_x)))

n=len(panels)
fig,axes=plt.subplots(1,n,figsize=(3.3*n,4.2))

for ax,(title,im) in zip(axes,panels):
    ax.imshow(im)
    ax.set_title(title,fontsize=15)
    ax.axis("off")

plt.tight_layout()
_out=os.path.join(RUN_DIR,"augmentation_techniques.png")
plt.savefig(_out,dpi=300,bbox_inches="tight")
plt.show()

print("sample image:",_ts)
print("techniques   :",", ".join(t for t,_ in _techniques))
print("saved figure :",_out)

In [ ]:
_APP=tf.keras.applications

_CANDIDATES=[
    ("vgg16","VGG16","vgg16"),("vgg19","VGG19","vgg19"),
    ("resnet50","ResNet50","resnet"),("resnet101","ResNet101","resnet"),("resnet152","ResNet152","resnet"),
    ("resnet50v2","ResNet50V2","resnet_v2"),("resnet101v2","ResNet101V2","resnet_v2"),("resnet152v2","ResNet152V2","resnet_v2"),
    ("inceptionv3","InceptionV3","inception_v3"),("inceptionresnetv2","InceptionResNetV2","inception_resnet_v2"),("xception","Xception","xception"),
    ("mobilenet","MobileNet","mobilenet"),("mobilenetv2","MobileNetV2","mobilenet_v2"),
    ("mobilenetv3small","MobileNetV3Small","mobilenet_v3"),("mobilenetv3large","MobileNetV3Large","mobilenet_v3"),
    ("densenet121","DenseNet121","densenet"),("densenet169","DenseNet169","densenet"),("densenet201","DenseNet201","densenet"),
    ("nasnetmobile","NASNetMobile","nasnet"),("nasnetlarge","NASNetLarge","nasnet"),
    ("efficientnetb0","EfficientNetB0","efficientnet"),("efficientnetb1","EfficientNetB1","efficientnet"),
    ("efficientnetb2","EfficientNetB2","efficientnet"),("efficientnetb3","EfficientNetB3","efficientnet"),
    ("efficientnetb4","EfficientNetB4","efficientnet"),("efficientnetb5","EfficientNetB5","efficientnet"),
    ("efficientnetb6","EfficientNetB6","efficientnet"),("efficientnetb7","EfficientNetB7","efficientnet"),
    ("efficientnetv2b0","EfficientNetV2B0","efficientnet_v2"),("efficientnetv2b1","EfficientNetV2B1","efficientnet_v2"),
    ("efficientnetv2b2","EfficientNetV2B2","efficientnet_v2"),("efficientnetv2b3","EfficientNetV2B3","efficientnet_v2"),
    ("efficientnetv2s","EfficientNetV2S","efficientnet_v2"),("efficientnetv2m","EfficientNetV2M","efficientnet_v2"),
    ("efficientnetv2l","EfficientNetV2L","efficientnet_v2"),
    ("convnexttiny","ConvNeXtTiny","convnext"),("convnextsmall","ConvNeXtSmall","convnext"),
    ("convnextbase","ConvNeXtBase","convnext"),("convnextlarge","ConvNeXtLarge","convnext"),("convnextxlarge","ConvNeXtXLarge","convnext"),
    ("regnetx002","RegNetX002","regnet"),("regnetx004","RegNetX004","regnet"),("regnetx006","RegNetX006","regnet"),
    ("regnetx008","RegNetX008","regnet"),("regnetx016","RegNetX016","regnet"),("regnetx032","RegNetX032","regnet"),
    ("regnetx040","RegNetX040","regnet"),("regnetx064","RegNetX064","regnet"),("regnetx080","RegNetX080","regnet"),
    ("regnetx120","RegNetX120","regnet"),("regnetx160","RegNetX160","regnet"),("regnetx320","RegNetX320","regnet"),
    ("regnety002","RegNetY002","regnet"),("regnety004","RegNetY004","regnet"),("regnety006","RegNetY006","regnet"),
    ("regnety008","RegNetY008","regnet"),("regnety016","RegNetY016","regnet"),("regnety032","RegNetY032","regnet"),
    ("regnety040","RegNetY040","regnet"),("regnety064","RegNetY064","regnet"),("regnety080","RegNetY080","regnet"),
    ("regnety120","RegNetY120","regnet"),("regnety160","RegNetY160","regnet"),("regnety320","RegNetY320","regnet")
]

MODEL_INPUT_SIZE={"nasnetlarge":331}

CNN_REGISTRY={}
for _name,_cls,_mod in _CANDIDATES:
    try:
        ctor=getattr(_APP,_cls)
        pre=getattr(getattr(_APP,_mod),"preprocess_input")
        CNN_REGISTRY[_name]=(ctor,pre)
    except Exception:
        pass

print(f"Available CNN backbones ({len(CNN_REGISTRY)}):")
for i,k in enumerate(CNN_REGISTRY.keys(),1):
    print(f"  {k:20s}",end="\n" if i%4==0 else "")
print()

In [ ]:
def _get_register():
    for _get in (lambda: keras.saving.register_keras_serializable(),lambda: tf.keras.utils.register_keras_serializable()):
        try:return _get()
        except Exception:continue
    return lambda _c:_c
register=_get_register()

PROJ_DIM=768
VIT_DEPTH=4
VIT_HEADS=8
VIT_MLP=2048
HEAD_DROPOUT=0.2

@register
class PrependCLS(layers.Layer):
    def __init__(self,dim,**kw):
        super().__init__(**kw);self.dim=dim
    def build(self,s):
        self.cls=self.add_weight(shape=(1,1,self.dim),initializer="random_normal",trainable=True,name="cls")
        super().build(s)
    def call(self,x):
        b=tf.shape(x)[0]
        cls=tf.broadcast_to(tf.cast(self.cls,x.dtype),[b,1,tf.shape(x)[-1]])
        return tf.concat([cls,x],axis=1)
    def get_config(self):
        c=super().get_config();c.update(dict(dim=self.dim));return c

@register
class AddPositionalEmbedding(layers.Layer):
    def build(self,s):
        self.pos=self.add_weight(shape=(1,int(s[1]),int(s[-1])),initializer="random_normal",trainable=True,name="pos")
        super().build(s)
    def call(self,x):
        return x+self.pos

@register
class TakeToken(layers.Layer):
    def __init__(self,index=0,**kw):
        super().__init__(**kw);self.index=index
    def call(self,x):
        return x[:,self.index]
    def get_config(self):
        c=super().get_config();c.update(dict(index=self.index));return c

@register
class TransformerEncoderBlock(layers.Layer):
    def __init__(self,dim,heads,mlp_dim,dropout=0.0,**kw):
        super().__init__(**kw);self.dim,self.heads,self.mlp_dim,self.dr=dim,heads,mlp_dim,dropout
        self.n1=layers.LayerNormalization(epsilon=1e-6)
        self.attn=layers.MultiHeadAttention(num_heads=heads,key_dim=dim//heads,dropout=dropout)
        self.n2=layers.LayerNormalization(epsilon=1e-6)
        self.m1=layers.Dense(mlp_dim,activation="gelu");self.m2=layers.Dense(dim)
        self.dp=layers.Dropout(dropout)
    def call(self,x,training=False):
        h=self.n1(x);x=x+self.attn(h,h,training=training)
        h=self.n2(x);h=self.dp(self.m2(self.dp(self.m1(h),training=training)),training=training)
        return x+h
    def get_config(self):
        c=super().get_config();c.update(dict(dim=self.dim,heads=self.heads,mlp_dim=self.mlp_dim,dropout=self.dr));return c

def build_agrinet27(num_classes):
    P=PROJ_DIM
    inp=keras.Input((IMG_SIZE,IMG_SIZE,3),name="input_image")
    x=data_augmentation(inp)

    backbone=keras.applications.EfficientNetV2B0(include_top=False,weights="imagenet",input_shape=(IMG_SIZE,IMG_SIZE,3))
    backbone.trainable=True
    cnn_feats=backbone(x,training=False)

    high=layers.Conv2D(P,1,name="proj_high")(cnn_feats)
    high_pool=layers.GlobalAveragePooling2D(name="high_pool")(high)

    mid=layers.AveragePooling2D(pool_size=2,strides=2,padding="same")(cnn_feats)
    mid=layers.Conv2D(P,1,name="proj_mid")(mid)
    mid_pool=layers.GlobalAveragePooling2D(name="mid_pool")(mid)

    low=layers.AveragePooling2D(pool_size=4,strides=4,padding="same")(cnn_feats)
    low=layers.Conv2D(P,1,name="proj_low")(low)
    low_pool=layers.GlobalAveragePooling2D(name="low_pool")(low)

    ms=layers.Concatenate(name="ms_concat")([high_pool,mid_pool,low_pool])
    cnn_emb=layers.Dense(P,activation="gelu",name="cnn_emb")(ms)
    cnn_query=layers.Reshape((1,P),name="cnn_query")(cnn_emb)

    tokens=layers.Reshape((7*7,P),name="to_tokens")(high)
    tokens=PrependCLS(P,name="cls_token")(tokens)
    tokens=AddPositionalEmbedding(name="pos_embed")(tokens)
    tokens=layers.Dropout(HEAD_DROPOUT)(tokens)

    for i in range(VIT_DEPTH):
        tokens=TransformerEncoderBlock(P,VIT_HEADS,VIT_MLP,HEAD_DROPOUT,name=f"vit_block{i}")(tokens)

    cls_rep=TakeToken(0,name="cls_rep")(tokens)

    attn=layers.MultiHeadAttention(num_heads=VIT_HEADS,key_dim=P//VIT_HEADS,name="cross_mha")(query=cnn_query,value=tokens,key=tokens)
    attn_fused=layers.Flatten(name="attn_fused")(attn)

    fused=layers.Concatenate(name="fusion_concat")([attn_fused,cls_rep,cnn_emb])
    r=layers.LayerNormalization(epsilon=1e-6,name="fusion_ln")(fused)
    r=layers.Dense(P,activation="gelu")(r)
    r=layers.Dropout(HEAD_DROPOUT)(r)
    out=layers.Dense(num_classes,activation="softmax",dtype="float32",name="output")(r)

    model=keras.Model(inp,out,name="AgriNet27")
    return model,backbone

model,backbone=build_agrinet27(NUM_CLASSES)
print("Built:",model.name)
model.summary()

In [ ]:
model.compile(
    optimizer=keras.optimizers.AdamW(learning_rate=LR, weight_decay=1e-5, clipnorm=CLIPNORM),
    loss=keras.losses.CategoricalCrossentropy(label_smoothing=LABEL_SMOOTHING),
    metrics=[
        "accuracy",
        keras.metrics.Precision(name="precision"),
        keras.metrics.Recall(name="recall"),
        keras.metrics.AUC(name="auc"),
    ],
)


In [ ]:
BEST_MODEL_PATH = os.path.join(RUN_DIR, "best_model.keras")

class TimeHistory(keras.callbacks.Callback):
    """Logs each epoch's wall-clock time to a jsonl file (survives resume)."""
    def __init__(self, log_path):
        super().__init__()
        self.log_path = log_path

    def on_epoch_begin(self, epoch, logs=None):
        self._t0 = time.time()

    def on_epoch_end(self, epoch, logs=None):
        with open(self.log_path, "a") as f:
            f.write(json.dumps({"epoch": int(epoch), "seconds": float(time.time() - self._t0)}) + "\n")


def make_callbacks(stage):
    """Per-model, per-stage. BackupAndRestore = auto mid-run resume."""
    backup_dir = os.path.join(CKPT_DIR, f"backup_{MODEL_NAME}_s{stage}")
    log_csv    = os.path.join(RUN_DIR, f"training_log_stage{stage}.csv")
    time_log   = os.path.join(CKPT_DIR, f"epoch_times_{MODEL_NAME}_s{stage}.jsonl")
    return [
        keras.callbacks.BackupAndRestore(backup_dir=backup_dir),
        keras.callbacks.ModelCheckpoint(BEST_MODEL_PATH, monitor="val_auc",
                                        save_best_only=True, mode="max", verbose=1),
        keras.callbacks.EarlyStopping(monitor="val_auc", mode="max", patience=12,
                                      restore_best_weights=True, verbose=1),
        keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5,
                                          patience=4, min_lr=1e-7, verbose=1),
        keras.callbacks.CSVLogger(log_csv, append=True),
        TimeHistory(time_log),
    ]

In [ ]:
STAGE1     = f"train_s1::{MODEL_NAME}"
W1         = os.path.join(RUN_DIR, "stage1.weights.h5")
H1_PATH    = os.path.join(CKPT_DIR, f"history_{MODEL_NAME}_s1.json")

if stage_done(STAGE1) and os.path.exists(W1):
    print(f"[resume] '{MODEL_NAME}' stage-1 done -- loading weights, skipping.")
    model.load_weights(W1)
    history_stage1 = load_json(H1_PATH) or {}
else:
    _h1 = model.fit(
        train_ds, validation_data=val_ds,
        epochs=EPOCHS,
        callbacks=make_callbacks(stage=1),
        class_weight=class_weight,
    )
    history_stage1 = _h1.history
    save_json(history_stage1, H1_PATH)
    model.save_weights(W1)
    mark_done(STAGE1)

In [ ]:
import os, json, time
import numpy as np
import pandas as pd
import tensorflow as tf

_rows = []
def _row(k, v): _rows.append((k, v))

n_total    = int(sum(np.prod(w.shape) for w in model.weights))
n_trainbl  = int(sum(np.prod(w.shape) for w in model.trainable_weights))
_row("Total parameters",        f"{n_total:,}")
_row("Trainable parameters",    f"{n_trainbl:,}")
_row("Non-trainable parameters",f"{n_total - n_trainbl:,}")
_row("Model size (fp32)",       f"{n_total * 4 / 1e6:.2f} MB")
_row("Model size (fp16)",       f"{n_total * 2 / 1e6:.2f} MB")

def _get_flops(m):
    try:
        from tensorflow.python.profiler.model_analyzer import profile
        from tensorflow.python.profiler.option_builder import ProfileOptionBuilder
        from tensorflow.python.framework.convert_to_constants import convert_variables_to_constants_v2
        spec = tf.TensorSpec([1, IMG_SIZE, IMG_SIZE, 3], tf.float32)
        cf = tf.function(lambda x: m(x, training=False)).get_concrete_function(spec)
        frozen = convert_variables_to_constants_v2(cf)
        opts = ProfileOptionBuilder.float_operation(); opts["output"] = "none"
        return profile(frozen.graph, options=opts).total_float_ops
    except Exception:
        return None
_flops = _get_flops(model)
if _flops:
    _row("FLOPs / image", f"{_flops / 1e9:.2f} GFLOPs")
    _row("MACs / image",  f"{_flops / 2 / 1e9:.2f} GMACs")
else:
    _row("FLOPs / image", "n/a (custom ops not traceable)")

def _epoch_times(stage):
    p = os.path.join(CKPT_DIR, f"epoch_times_{MODEL_NAME}_s{stage}.jsonl")
    out = []
    if os.path.exists(p):
        for ln in open(p):
            ln = ln.strip()
            if ln:
                try: out.append(json.loads(ln)["seconds"])
                except Exception: pass
    return out
_t1, _t2 = _epoch_times(1), _epoch_times(2)
_all = _t1 + _t2
_n_train_imgs = sum(len(os.listdir(os.path.join(train_dir, c)))
                    for c in os.listdir(train_dir) if os.path.isdir(os.path.join(train_dir, c)))
_row("Training images", f"{_n_train_imgs:,}")
if _all:
    _avg = sum(_all) / len(_all)
    _row("Epochs run",          f"{len(_all)} (stage1: {len(_t1)}, stage2: {len(_t2)})")
    _row("Avg epoch time",      f"{_avg:.1f} s")
    _row("Total training time", f"{sum(_all) / 60:.1f} min")
    if _avg > 0 and _n_train_imgs:
        _row("Train throughput", f"{_n_train_imgs / _avg:.1f} img/s")
else:
    _row("Training time", "n/a (not trained / resumed this session)")

def _bench(device, reps=20, warmup=5):
    try:
        with tf.device(device):
            xb = tf.random.uniform([1, IMG_SIZE, IMG_SIZE, 3])
            for _ in range(warmup): _ = model(xb, training=False)
            if device.endswith("GPU:0"):
                try: tf.keras.backend.clear_session
                except Exception: pass
            t0 = time.time()
            for _ in range(reps): _ = model(xb, training=False)
            return (time.time() - t0) / reps * 1000.0
    except Exception:
        return None
_gpu = len(tf.config.list_physical_devices('GPU')) > 0
_dev = "/GPU:0" if _gpu else "/CPU:0"
_lat = _bench(_dev)
if _lat:
    _row(f"Inference latency ({'GPU' if _gpu else 'CPU'})",     f"{_lat:.2f} ms/image")
    _row(f"Inference throughput ({'GPU' if _gpu else 'CPU'})",  f"{1000.0 / _lat:.1f} img/s")
if _gpu:                                   # CPU = edge-device simulation
    _lat_cpu = _bench("/CPU:0", reps=5, warmup=2)
    if _lat_cpu: _row("Inference latency (CPU, edge sim.)", f"{_lat_cpu:.2f} ms/image")

if _gpu:
    try:
        _row("Peak GPU memory", f"{tf.config.experimental.get_memory_info('GPU:0')['peak'] / 1e6:.0f} MB")
    except Exception:
        pass

cost_df = pd.DataFrame(_rows, columns=["Metric", f"Value ({MODEL_NAME})"])
_out = os.path.join(RUN_DIR, "computational_cost.csv")
cost_df.to_csv(_out, index=False)
print(f"===== Computational Cost — {MODEL_NAME} =====")
print(cost_df.to_string(index=False))
print("saved:", _out)
try:
    from IPython.display import display
    display(cost_df)
except Exception:
    pass


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def _as_hist(h):
    if h is None:return {}
    return h.history if hasattr(h,"history") else h

def _smooth(y,frac=0.15):
    y=np.asarray(y,dtype=float)
    n=len(y)
    if n<4:return y
    w=max(3,int(round(n*frac))|1)
    pad=w//2
    yp=np.pad(y,(pad,pad),mode="edge")
    kernel=np.ones(w)/w
    return np.convolve(yp,kernel,mode="valid")[:n]

def _panel(ax,tr,val,title,ylabel,best_mode):
    ep=np.arange(1,len(tr)+1)
    ax.plot(ep,tr,color="#4C9BE8",lw=1,alpha=0.30)
    ax.plot(ep,val,color="#E8A24C",lw=1,alpha=0.30)
    ax.plot(ep,_smooth(tr),color="#2ca02c",lw=2.5,label=f"Train {ylabel}")
    ax.plot(ep,_smooth(val),color="#d62728",lw=2.5,ls="--",label=f"Validation {ylabel}")
    if len(val):
        be=int(np.argmin(val) if best_mode=="min" else np.argmax(val))
        ax.axvline(be+1,color="#1f6fb2",ls=":",lw=1.5,alpha=0.9)
        ax.scatter([be+1],[val[be]],color="#1f6fb2",s=60,zorder=5)
    ax.set_title(title,fontsize=13,fontweight="bold")
    ax.set_xlabel("Epoch")
    ax.set_ylabel(ylabel)
    ax.grid(True,ls="--",alpha=0.35)
    ax.legend(frameon=True,fontsize=10)

def plot_history(h1,h2=None,model_label=None):
    h1=_as_hist(h1)
    h2=_as_hist(h2)
    acc=list(h1.get('accuracy',[]))+list(h2.get('accuracy',[]))
    vacc=list(h1.get('val_accuracy',[]))+list(h2.get('val_accuracy',[]))
    loss=list(h1.get('loss',[]))+list(h2.get('loss',[]))
    vloss=list(h1.get('val_loss',[]))+list(h2.get('val_loss',[]))
    tag=f" ({model_label})" if model_label else (f" ({MODEL_NAME})" if "MODEL_NAME" in globals() else "")

    fig,(a1,a2)=plt.subplots(1,2,figsize=(13,5))
    _panel(a1,loss,vloss,"Training vs Validation Loss"+tag,"Loss","min")
    _panel(a2,acc,vacc,"Training vs Validation Accuracy"+tag,"Accuracy","max")
    plt.tight_layout()
    out=os.path.join(RUN_DIR,"training_curves.png") if "RUN_DIR" in globals() else "training_curves.png"
    plt.savefig(out,dpi=300,bbox_inches="tight")
    plt.show()
    print("saved:",out)

try:
    plot_history(history_stage1,history_stage2)
except NameError:
    plot_history(history)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def _as_hist(h):
    if h is None:return {}
    return h.history if hasattr(h,"history") else h

def _smooth(y,frac=0.15):
    y=np.asarray(y,dtype=float);n=len(y)
    if n<4:return y
    w=max(3,int(round(n*frac))|1);pad=w//2
    yp=np.pad(y,(pad,pad),mode="edge")
    return np.convolve(yp,np.ones(w)/w,mode="valid")[:n]

def plot_metric_curve(h1,h2,metric_name,display_name):
    h1,h2=_as_hist(h1),_as_hist(h2)
    tr=list(h1.get(metric_name,[]))+list(h2.get(metric_name,[]))
    val=list(h1.get("val_"+metric_name,[]))+list(h2.get("val_"+metric_name,[]))
    ep=np.arange(1,len(tr)+1)
    tag=f" ({MODEL_NAME})" if "MODEL_NAME" in globals() else ""

    plt.figure(figsize=(7,5))
    ax=plt.gca()
    ax.plot(ep,tr,color="#4C9BE8",lw=1,alpha=0.30)
    ax.plot(ep,val,color="#E8A24C",lw=1,alpha=0.30)
    ax.plot(ep,_smooth(tr),color="#2ca02c",lw=2.5,label=f"Train {display_name}")
    ax.plot(ep,_smooth(val),color="#d62728",lw=2.5,ls="--",label=f"Validation {display_name}")

    if len(val):
        be=int(np.argmax(val))
        ax.axvline(be+1,color="#1f6fb2",ls=":",lw=1.5,alpha=0.9)
        ax.scatter([be+1],[val[be]],color="#1f6fb2",s=60,zorder=5)

    ax.set_title(f"Training vs Validation {display_name}"+tag,fontsize=13,fontweight="bold")
    ax.set_xlabel("Epoch")
    ax.set_ylabel(display_name)
    ax.grid(True,ls="--",alpha=0.35)
    ax.legend(frameon=True,fontsize=10)

    plt.tight_layout()
    out=os.path.join(RUN_DIR,f"{metric_name}_curve.png") if "RUN_DIR" in globals() else f"{metric_name}_curve.png"
    plt.savefig(out,dpi=300,bbox_inches="tight")
    plt.show()
    print("saved:",out)

plot_metric_curve(history_stage1,history_stage2,"precision","Precision")
plot_metric_curve(history_stage1,history_stage2,"recall","Recall")

In [ ]:
test_loss,test_acc,test_prec,test_rec,test_auc=model.evaluate(test_ds)
print(f"Test Accuracy : {test_acc:.4f}")
print(f"Test Precision: {test_prec:.4f}")
print(f"Test Recall   : {test_rec:.4f}")
print(f"Test AUC      : {test_auc:.4f}")

y_true,y_pred,y_prob=[],[],[]

for images,labels in test_ds:
    preds=model.predict(images,verbose=0)
    y_true.extend(np.argmax(labels.numpy(),axis=1))
    y_pred.extend(np.argmax(preds,axis=1))
    y_prob.extend(preds)

y_true=np.array(y_true)
y_pred=np.array(y_pred)
y_prob=np.array(y_prob)

In [ ]:
cm=confusion_matrix(y_true,y_pred)
cm_norm=confusion_matrix(y_true,y_pred,normalize="true")

FS=16

def _plot_cm(matrix,title,fmt,fname):
    plt.figure(figsize=(23,21))
    ax=plt.gca()
    sns.heatmap(
        matrix,
        annot=True,
        fmt=fmt,
        cmap="Blues",
        xticklabels=CLASS_NAMES,
        yticklabels=CLASS_NAMES,
        ax=ax,
        annot_kws={"size":FS},
        cbar_kws={"shrink":0.8}
    )
    ax.set_title(title,fontsize=FS,fontweight="bold")
    ax.set_xlabel("Predicted",fontsize=FS)
    ax.set_ylabel("True",fontsize=FS)
    ax.tick_params(axis="both",labelsize=FS)
    ax.set_xticklabels(ax.get_xticklabels(),rotation=45,ha="right",fontsize=FS)
    ax.set_yticklabels(ax.get_yticklabels(),rotation=0,fontsize=FS)
    ax.collections[0].colorbar.ax.tick_params(labelsize=FS)
    plt.tight_layout()
    out=os.path.join(RUN_DIR,fname)
    plt.savefig(out,dpi=300,bbox_inches="tight")
    plt.show()
    print("saved:",out)

_plot_cm(cm,"Confusion Matrix (counts)","d","confusion_matrix_counts.png")
_plot_cm(cm_norm,"Confusion Matrix (row-normalized)",".2f","confusion_matrix_normalized.png")

In [ ]:
report_dict = classification_report(y_true, y_pred, target_names=CLASS_NAMES, output_dict=True)
report_df = pd.DataFrame(report_dict).transpose()
report_df.to_csv(os.path.join(RUN_DIR, "classification_report.csv"))

print("Class-wise Classification Report:\n")
print(classification_report(y_true, y_pred, target_names=CLASS_NAMES))

report_df


In [ ]:
from sklearn.preprocessing import label_binarize
from sklearn.metrics import precision_recall_curve,average_precision_score
import numpy as np
import matplotlib.pyplot as plt

FS=16

y_true_bin=label_binarize(y_true,classes=list(range(NUM_CLASSES)))
if y_true_bin.shape[1]==1:
    y_true_bin=np.hstack([1-y_true_bin,y_true_bin])

plt.figure(figsize=(10,8))
ax=plt.gca()
cmap=plt.get_cmap("tab10" if NUM_CLASSES<=10 else "tab20")

for i,class_name in enumerate(CLASS_NAMES):
    p,r,_=precision_recall_curve(y_true_bin[:,i],y_prob[:,i])
    ap=average_precision_score(y_true_bin[:,i],y_prob[:,i])
    ax.plot(
        r,p,lw=2,
        color=cmap(i%10 if NUM_CLASSES<=10 else i%20),
        label=f"{class_name} (AP={ap:.2f})"
    )

p_m,r_m,_=precision_recall_curve(y_true_bin.ravel(),y_prob.ravel())
ap_m=average_precision_score(y_true_bin,y_prob,average="micro")
ax.plot(
    r_m,p_m,lw=3.5,color="black",ls="--",
    label=f"Micro-average (AP={ap_m:.2f})"
)

ax.set_xlabel("Recall",fontsize=FS)
ax.set_ylabel("Precision",fontsize=FS)
ax.set_title("Precision-Recall Curve (per class, one-vs-rest)",fontsize=FS,fontweight="bold")
ax.tick_params(axis="both",labelsize=FS)
ax.set_xlim(-0.02,1.02)
ax.set_ylim(-0.02,1.02)
ax.grid(True,ls="--",alpha=0.35)
ax.legend(loc="upper left",bbox_to_anchor=(1.02,1.0),fontsize=FS,frameon=True)

plt.tight_layout()
_out=os.path.join(RUN_DIR,"precision_recall_curve.png")
plt.savefig(_out,dpi=300,bbox_inches="tight")
plt.show()
print("saved:",_out)

In [ ]:
from sklearn.preprocessing import label_binarize
from sklearn.metrics import roc_curve,auc
import numpy as np
import matplotlib.pyplot as plt

FS=16

y_true_bin=label_binarize(y_true,classes=list(range(NUM_CLASSES)))
if y_true_bin.shape[1]==1:
    y_true_bin=np.hstack([1-y_true_bin,y_true_bin])

fpr,tpr,roc_auc={},{},{}
for i in range(NUM_CLASSES):
    fpr[i],tpr[i],_=roc_curve(y_true_bin[:,i],y_prob[:,i])
    roc_auc[i]=auc(fpr[i],tpr[i])

fpr_micro,tpr_micro,_=roc_curve(y_true_bin.ravel(),y_prob.ravel())
auc_micro=auc(fpr_micro,tpr_micro)

all_fpr=np.unique(np.concatenate([fpr[i] for i in range(NUM_CLASSES)]))
mean_tpr=np.zeros_like(all_fpr)
for i in range(NUM_CLASSES):
    mean_tpr+=np.interp(all_fpr,fpr[i],tpr[i])
mean_tpr/=NUM_CLASSES
auc_macro=auc(all_fpr,mean_tpr)

plt.figure(figsize=(10,8))
ax=plt.gca()
cmap=plt.get_cmap("tab10" if NUM_CLASSES<=10 else "tab20")

for i,class_name in enumerate(CLASS_NAMES):
    ax.plot(
        fpr[i],tpr[i],lw=2,
        color=cmap(i%10 if NUM_CLASSES<=10 else i%20),
        label=f"{class_name} (AUC={roc_auc[i]:.2f})"
    )

ax.plot(fpr_micro,tpr_micro,lw=3.5,color="black",ls="--",label=f"Micro-average (AUC={auc_micro:.2f})")
ax.plot(all_fpr,mean_tpr,lw=3.5,color="dimgray",ls=":",label=f"Macro-average (AUC={auc_macro:.2f})")
ax.plot([0,1],[0,1],lw=1.5,color="red",ls="--",alpha=0.6,label="Chance")

ax.set_xlabel("False Positive Rate",fontsize=FS)
ax.set_ylabel("True Positive Rate",fontsize=FS)
ax.set_title("ROC Curve (per class, one-vs-rest)",fontsize=FS,fontweight="bold")
ax.tick_params(axis="both",labelsize=FS)
ax.set_xlim(-0.02,1.02)
ax.set_ylim(-0.02,1.02)
ax.grid(True,ls="--",alpha=0.35)
ax.legend(loc="upper left",bbox_to_anchor=(1.02,1.0),fontsize=FS,frameon=True)

plt.tight_layout()
_out=os.path.join(RUN_DIR,"roc_curve.png")
plt.savefig(_out,dpi=300,bbox_inches="tight")
plt.show()
print("saved:",_out)

In [ ]:
from sklearn.metrics import cohen_kappa_score,matthews_corrcoef,accuracy_score,f1_score

_kappa=cohen_kappa_score(y_true,y_pred)
_mcc=matthews_corrcoef(y_true,y_pred)
_acc=accuracy_score(y_true,y_pred)
_f1m=f1_score(y_true,y_pred,average="macro")

print(f"===== {MODEL_NAME} — chance-corrected metrics =====")
print(f"Accuracy            : {_acc:.4f}")
print(f"Macro F1            : {_f1m:.4f}")
print(f"Cohen's Kappa       : {_kappa:.4f}")
print(f"Matthews Corr (MCC) : {_mcc:.4f}")

import numpy as np,json

_pred_path=os.path.join(RUN_DIR,f"test_predictions_{MODEL_NAME}.npz")
np.savez(
    _pred_path,
    y_true=np.asarray(y_true),
    y_pred=np.asarray(y_pred),
    y_prob=np.asarray(y_prob)
)

with open(os.path.join(RUN_DIR,f"stat_metrics_{MODEL_NAME}.json"),"w") as _f:
    json.dump({
        "model":MODEL_NAME,
        "accuracy":float(_acc),
        "macro_f1":float(_f1m),
        "cohen_kappa":float(_kappa),
        "mcc":float(_mcc)
    },_f,indent=2)

print("saved predictions ->",_pred_path)

In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score,f1_score

_yt=np.asarray(y_true)
_yp=np.asarray(y_pred)
N_BOOT=2000
rng=np.random.default_rng(42)
_acc_bs,_f1_bs=[],[]
_n=len(_yt)

for _ in range(N_BOOT):
    idx=rng.integers(0,_n,_n)
    _acc_bs.append(accuracy_score(_yt[idx],_yp[idx]))
    _f1_bs.append(f1_score(_yt[idx],_yp[idx],average="macro"))

def _ci(vals):
    lo,hi=np.percentile(vals,[2.5,97.5])
    return float(np.mean(vals)),float(lo),float(hi)

_a_m,_a_lo,_a_hi=_ci(_acc_bs)
_f_m,_f_lo,_f_hi=_ci(_f1_bs)

print(f"===== {MODEL_NAME} — Bootstrap 95% CI ({N_BOOT} resamples) =====")
print(f"Accuracy : {_a_m:.4f}  (95% CI: {_a_lo:.4f} - {_a_hi:.4f})")
print(f"Macro F1 : {_f_m:.4f}  (95% CI: {_f_lo:.4f} - {_f_hi:.4f})")

In [ ]:
final_path = os.path.join(RUN_DIR, f"cnn_{MODEL_NAME}_final.keras")
model.save(final_path)
mark_done(f"save::{MODEL_NAME}", {"path": final_path})
print("Saved:", final_path)
print(f"All outputs for '{MODEL_NAME}' are under: {RUN_DIR}")
print("Switch MODEL_NAME (Config) and re-run to train/compare another backbone.")


In [ ]:
import os,json,numpy as np
from sklearn.metrics import accuracy_score,precision_score,recall_score,f1_score,cohen_kappa_score,matthews_corrcoef

N_BOOT=2000
CI=95
_lo_q,_hi_q=(100-CI)/2,100-(100-CI)/2

rng=np.random.default_rng(42)
yt=np.asarray(y_true);yp=np.asarray(y_pred);n=len(yt)

def _ci(fn,**kw):
    vals=np.empty(N_BOOT)
    for b in range(N_BOOT):
        idx=rng.integers(0,n,n)
        vals[b]=fn(yt[idx],yp[idx],**kw)
    point=fn(yt,yp,**kw)
    lo,hi=np.percentile(vals,[_lo_q,_hi_q])
    return float(point),float(lo),float(hi)

rows={
    "Accuracy":_ci(accuracy_score),
    "Precision (macro)":_ci(precision_score,average="macro",zero_division=0),
    "Recall (macro)":_ci(recall_score,average="macro",zero_division=0),
    "F1-score (macro)":_ci(f1_score,average="macro",zero_division=0),
    "Cohen's Kappa":_ci(cohen_kappa_score),
    "MCC":_ci(matthews_corrcoef)
}

print(f"===== {CI}% Confidence Intervals (Bootstrap, N={N_BOOT}, {n} test samples) =====")
print(f"{'metric':<20}{'value':>9}{'':3}{f'{CI}% CI':>22}")

for k,(p,lo,hi) in rows.items():
    print(f"{k:<20}{p*100:>8.2f}%   [{lo*100:6.2f}%, {hi*100:6.2f}%]")

_out=os.path.join(RUN_DIR,"confidence_intervals.json") if "RUN_DIR" in globals() else "confidence_intervals.json"

json.dump(
    {k:{"value":p,"ci_low":lo,"ci_high":hi} for k,(p,lo,hi) in rows.items()},
    open(_out,"w"),
    indent=2
)

print("saved:",_out)
print(
    f"\nReport, e.g.:  Accuracy = {rows['Accuracy'][0]*100:.2f}% "
    f"(95% CI: {rows['Accuracy'][1]*100:.2f}-{rows['Accuracy'][2]*100:.2f}%)"
)

In [ ]:
import os,collections,numpy as np,tensorflow as tf
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score,precision_score,recall_score,f1_score

CV_FOLDS=5
CV_EPOCHS=30
CV_BATCH=64
APPLY_PREP=True

assert "build_agrinet27" in globals(),"Run the model BUILD cell first (defines build_agrinet27)."
BUILD_FN=build_agrinet27

assert "DATA_DIR" in globals(),"Set DATA_DIR."
_EXTS=(".png",".jpg",".jpeg",".bmp",".tif",".tiff",".webp")
_all=[]

for root,_dirs,files in os.walk(DATA_DIR):
    for f in files:
        if f.lower().endswith(_EXTS):
            _all.append((os.path.join(root,f),os.path.basename(root)))

assert _all,f"No images found under DATA_DIR={DATA_DIR}"
_disc=sorted(set(cn for _,cn in _all))
classes=list(CLASS_NAMES) if ("CLASS_NAMES" in globals() and set(CLASS_NAMES)==set(_disc)) else _disc
cls_to_idx={c:i for i,c in enumerate(classes)}
paths=np.array([p for p,cn in _all if cn in cls_to_idx])
labels=np.array([cls_to_idx[cn] for _,cn in _all if cn in cls_to_idx])
NUM_CLASSES=len(classes)
_cnt=collections.Counter(labels.tolist())

print(f"Found {len(paths)} images across {len(classes)} classes (pooled) for {CV_FOLDS}-fold CV.")
_few=[classes[i] for i in _cnt if _cnt[i]<CV_FOLDS]
assert not _few,f"Classes with < {CV_FOLDS} images (can't stratify): {_few}"

def _np_prep(p):
    import cv2
    bgr=cv2.imread(p.decode())
    bgr=cv2.resize(bgr,(IMG_SIZE,IMG_SIZE))
    if APPLY_PREP and "apply_preprocessing_bgr" in globals():
        bgr=apply_preprocessing_bgr(bgr)
    return cv2.cvtColor(bgr,cv2.COLOR_BGR2RGB).astype("float32")

def _load(path,label):
    img=tf.numpy_function(_np_prep,[path],tf.float32)
    img.set_shape((IMG_SIZE,IMG_SIZE,3))
    return img,tf.one_hot(label,NUM_CLASSES)

def _ds(idx,training):
    ds=tf.data.Dataset.from_tensor_slices((paths[idx],labels[idx]))
    if training:
        ds=ds.shuffle(min(len(idx),1000))
    return ds.map(_load,num_parallel_calls=tf.data.AUTOTUNE).batch(CV_BATCH).prefetch(tf.data.AUTOTUNE)

skf=StratifiedKFold(n_splits=CV_FOLDS,shuffle=True,random_state=42)
cv_results={"Accuracy":[],"Precision(macro)":[],"Recall(macro)":[],"F1(macro)":[]}

for fold,(tr,va) in enumerate(skf.split(paths,labels),1):
    print(f"\n----- Fold {fold}/{CV_FOLDS}  (train={len(tr)}, val={len(va)}) -----")
    tf.keras.backend.clear_session()
    m,_=BUILD_FN(NUM_CLASSES)
    m.compile(
        optimizer=tf.keras.optimizers.Adam(1e-4),
        loss="categorical_crossentropy",
        metrics=["accuracy"]
    )
    m.fit(_ds(tr,True),validation_data=_ds(va,False),epochs=CV_EPOCHS,verbose=2)
    prob=m.predict(_ds(va,False),verbose=0)
    yp=np.argmax(prob,1)
    yt=labels[va]

    cv_results["Accuracy"].append(accuracy_score(yt,yp))
    cv_results["Precision(macro)"].append(precision_score(yt,yp,average="macro",zero_division=0))
    cv_results["Recall(macro)"].append(recall_score(yt,yp,average="macro",zero_division=0))
    cv_results["F1(macro)"].append(f1_score(yt,yp,average="macro",zero_division=0))

    print(f"Fold {fold}: acc={cv_results['Accuracy'][-1]*100:.2f}%  F1={cv_results['F1(macro)'][-1]*100:.2f}%")

print(f"\n===== 5-Fold CV summary (Mean +/- SD) =====")
print(f"{'metric':<18}{'mean':>9}{'SD':>9}{'   per-fold'}")

import json
summary={}

for k,v in cv_results.items():
    v=np.asarray(v)
    summary[k]=(float(v.mean()),float(v.std()))
    print(f"{k:<18}{v.mean()*100:>8.2f}%{v.std()*100:>8.2f}%   [{', '.join(f'{x*100:.2f}' for x in v)}]")

_out=os.path.join(RUN_DIR,"cross_validation.json") if "RUN_DIR" in globals() else "cross_validation.json"

json.dump(
    {
        "per_fold":cv_results,
        "mean_sd":{k:{"mean":m_,"sd":s_} for k,(m_,s_) in summary.items()}
    },
    open(_out,"w"),
    indent=2
)

print("saved:",_out)

In [ ]:
import os,numpy as np,tensorflow as tf,cv2,matplotlib.pyplot as plt

FS=16
names=CLASS_NAMES if "CLASS_NAMES" in globals() else None
N_PER_CLASS=1
LIME_SAMPLES=6
DOT_DENSITY=0.12
DOT_RADIUS=2
DOT_COLOR=(255,230,0)

CAM_LAYER="proj_high"
try:_tC=model.get_layer(CAM_LAYER).output
except Exception:
    _tC=None
    for l in reversed(model.layers):
        try:
            if len(l.output.shape)==4:
                _tC=l.output;CAM_LAYER=l.name;break
        except Exception:pass

_grad_model=tf.keras.Model(model.inputs,[_tC,model.output])

def _cam(img,cls,plus=False):
    x=tf.convert_to_tensor(img[None].astype("float32"))
    with tf.GradientTape() as tape:
        conv,preds=_grad_model(x,training=False);tape.watch(conv)
        loss=preds[:,cls]
    grads=tf.cast(tape.gradient(loss,conv),tf.float32)
    conv=tf.cast(conv,tf.float32)
    if plus:
        g2,g3=grads*grads,grads*grads*grads
        denom=2.0*g2+tf.reduce_sum(conv,axis=(0,1,2))[None,None,None,:]*g3+1e-8
        alpha=tf.where(grads!=0,g2/denom,0.0)
        w=tf.reduce_sum(alpha*tf.nn.relu(grads),axis=(0,1,2))
    else:
        w=tf.reduce_mean(grads,axis=(0,1,2))
    cam=tf.nn.relu(tf.reduce_sum(conv[0]*w,axis=-1)).numpy()
    cam=cam/(cam.max()+1e-8)
    return cv2.resize(cam.astype("float32"),(img.shape[1],img.shape[0]))

_tok_model=tf.keras.Model(model.inputs,model.get_layer("to_tokens").output)
_cls=model.get_layer("cls_token")
_pos=model.get_layer("pos_embed")
_blocks=[l for l in model.layers if l.name.startswith("vit_block")]

def _vit_attention(img):
    x=_tok_model(img[None].astype("float32"),training=False)
    x=_pos(_cls(x))
    atts=[]
    for blk in _blocks:
        h=blk.n1(x)
        ao,sc=blk.attn(h,h,return_attention_scores=True,training=False)
        x=x+ao
        h=blk.n2(x)
        x=x+blk.m2(blk.m1(h))
        atts.append(tf.cast(sc,tf.float32).numpy()[0].mean(0))
    N=atts[0].shape[0]
    roll=np.eye(N)
    for A in atts:
        A=A+np.eye(N)
        A=A/A.sum(-1,keepdims=True)
        roll=A@roll
    cls_to_patch=roll[0,1:]
    g=int(round(np.sqrt(len(cls_to_patch))))
    cam=cls_to_patch.reshape(g,g)
    cam=(cam-cam.min())/(cam.max()-cam.min()+1e-8)
    return cv2.resize(cam.astype("float32"),(img.shape[1],img.shape[0]))

def _lime(img,model_range_max=255.0):
    from lime import lime_image
    disp=np.clip(img/255.0,0,1) if img.max()>1.5 else np.clip(img,0,1)

    def _pf(imgs):
        a=np.asarray(imgs,"float32")
        if a.max()<=1.5:a=a*model_range_max
        return model.predict(a,verbose=0)

    exp=lime_image.LimeImageExplainer().explain_instance(
        disp.astype("double"),_pf,top_labels=1,hide_color=0,num_samples=LIME_SAMPLES
    )
    lbl=exp.top_labels[0]
    _,mask=exp.get_image_and_mask(
        lbl,positive_only=True,num_features=8,hide_rest=False
    )

    out=np.ascontiguousarray(np.uint8(disp*255))
    ys,xs=np.nonzero(mask)

    if len(ys)>0:
        rng=np.random.RandomState(0)
        n=max(1,int(len(ys)*DOT_DENSITY))
        idx=rng.choice(len(ys),size=min(n,len(ys)),replace=False)
        for i in idx:
            cv2.circle(out,(int(xs[i]),int(ys[i])),DOT_RADIUS,DOT_COLOR,-1)

    return out.astype("float32")/255.0

def _disp(img):
    im=np.asarray(img,"float32")
    return np.clip((im-im.min())/(im.max()-im.min()+1e-6),0,1)

def _overlay(img,cam):
    d=_disp(img)
    heat=cv2.applyColorMap(np.uint8(255*cam),cv2.COLORMAP_JET)[:,:,::-1]/255.0
    return np.clip(0.5*d+0.5*heat,0,1)

picked={}
rows=[]

for images,labels in test_ds:
    preds=model.predict(images,verbose=0)
    p=np.argmax(preds,1)
    t=np.argmax(labels.numpy(),1)

    for i in range(len(p)):
        c=int(t[i])
        if p[i]==c and picked.get(c,0)<N_PER_CLASS:
            picked[c]=picked.get(c,0)+1
            rows.append((images[i].numpy(),c,float(preds[i][c])))

    if names and all(picked.get(c,0)>=N_PER_CLASS for c in range(len(names))):break
    if not names and len(rows)>=10:break

col_titles=["Original Image","LIME Overlay","Grad-CAM","Grad-CAM++","ViT Attention","Prediction Score"]
R=len(rows)
fig,axes=plt.subplots(R,6,figsize=(6*2.7,R*2.7))

if R==1:axes=axes[None,:]

for r,(img,c,conf) in enumerate(rows):
    cn=names[c] if names else str(c)
    lime_img=_lime(img)
    gc=_cam(img,c,False)
    gcpp=_cam(img,c,True)
    va=_vit_attention(img)

    panels=[
        _disp(img),
        lime_img,
        _overlay(img,gc),
        _overlay(img,gcpp),
        _overlay(img,va)
    ]

    for j in range(5):
        axes[r,j].imshow(panels[j])
        axes[r,j].axis("off")
        if r==0:
            axes[r,j].set_title(col_titles[j],fontsize=FS)

    ax=axes[r,5]
    ax.bar([cn],[conf],color="#1f77b4",width=0.5)
    ax.set_ylim(0,1)
    ax.tick_params(labelsize=FS-2)
    ax.text(0,conf+0.02,f"{conf*100:.1f}%",ha="center",fontsize=FS-3)

    if r==0:
        ax.set_title(col_titles[5],fontsize=FS)

    axes[r,0].set_ylabel(
        cn,fontsize=FS,fontweight="bold",
        rotation=0,ha="right",va="center",labelpad=40
    )

plt.tight_layout()
_o=os.path.join(RUN_DIR,"xai_combined_panel.png") if "RUN_DIR" in globals() else "xai_combined_panel.png"
plt.savefig(_o,dpi=200,bbox_inches="tight")
plt.show()
print("saved:",_o)